<a href="https://colab.research.google.com/github/samikshanimje/SmartECG-HD/blob/main/notebooks/03_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
import os
import sys
import importlib

PROJECT_ROOT = "/content/drive/MyDrive/SmartECG-HD"

# Add project root to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project Root:", PROJECT_ROOT)
print("Exists:", os.path.exists(PROJECT_ROOT))
print("src Exists:", os.path.exists(os.path.join(PROJECT_ROOT, "src")))
print("sys.path[0]:", sys.path[0])

# Clear cached imports if any
sys.modules.pop("src", None)
sys.modules.pop("src.config", None)

importlib.invalidate_caches()

from src.config import *

print("✅ config imported successfully!")
print(RAW_DATA_DIR)

Project Root: /content/drive/MyDrive/SmartECG-HD
Exists: True
src Exists: True
sys.path[0]: /content/drive/MyDrive/SmartECG-HD
✅ config imported successfully!
/content/drive/MyDrive/SmartECG-HD/datasets/mitbih


In [26]:
%%writefile /content/drive/MyDrive/SmartECG-HD/src/__init__.py
# SmartECG-HD package

Writing /content/drive/MyDrive/SmartECG-HD/src/__init__.py


In [37]:
from src.config import *

In [33]:
import os

print("Current directory:", os.getcwd())

print("\nFiles here:")
print(os.listdir("."))

print("\nDoes src exist here?")
print(os.path.exists("src"))

print("\nDrive src exists?")
print(os.path.exists("/content/drive/MyDrive/SmartECG-HD/src"))

Current directory: /content

Files here:
['.config', 'drive', 'sample_data']

Does src exist here?
False

Drive src exists?
True


In [34]:
import os

print(os.listdir("/content/drive/MyDrive/SmartECG-HD/src"))

print("\n__init__.py exists?",
      os.path.exists("/content/drive/MyDrive/SmartECG-HD/src/__init__.py"))

['__pycache__', 'data_loader.py', 'config.py', 'preprocessing.py', 'save_utils.py', 'label_generator.py', 'dataset_builder.py', 'feature_engineering.py', 'model.py', '__init__.py']

__init__.py exists? True


In [32]:
import importlib.util

print(importlib.util.find_spec("src"))

None


In [40]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

In [38]:
import importlib

import src.model

importlib.reload(src.model)

from src.model import build_model

print("✅ Model imported")

✅ Model imported


In [41]:
beats = np.load(BEATS_PATH)["beats"]

labels = np.load(LABELS_PATH)

print(beats.shape)
print(labels.shape)

(109380, 300)
(109380,)


In [42]:
encoder = LabelEncoder()

y = encoder.fit_transform(labels)

y = to_categorical(y)

X = beats.reshape((-1, 300, 1))

print(X.shape)
print(y.shape)

(109380, 300, 1)
(109380, 5)


In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(87504, 300, 1)
(21876, 300, 1)


In [44]:
%%writefile /content/drive/MyDrive/SmartECG-HD/src/model.py

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    Dropout,
    Bidirectional,
    LSTM,
    Dense
)

from tensorflow.keras.models import Model


def build_model():

    inputs = Input(shape=(300,1))

    # -------------------------
    # CNN Block 1
    # -------------------------

    x = Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        padding="same"
    )(inputs)

    x = BatchNormalization()(x)

    x = MaxPooling1D(2)(x)

    x = Dropout(0.2)(x)

    # -------------------------
    # CNN Block 2
    # -------------------------

    x = Conv1D(
        128,
        kernel_size=5,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)

    x = MaxPooling1D(2)(x)

    x = Dropout(0.2)(x)

    # -------------------------
    # CNN Block 3
    # -------------------------

    x = Conv1D(
        256,
        kernel_size=3,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)

    x = MaxPooling1D(2)(x)

    x = Dropout(0.2)(x)

    # -------------------------
    # BiLSTM
    # -------------------------

    x = Bidirectional(
        LSTM(
            64
        )
    )(x)

    # -------------------------
    # Dense
    # -------------------------

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(0.4)(x)

    outputs = Dense(
        5,
        activation="softmax"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )

    return model

Overwriting /content/drive/MyDrive/SmartECG-HD/src/model.py


In [45]:
import importlib

import src.model

importlib.reload(src.model)

from src.model import build_model

In [46]:
model = build_model()

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 37, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 37, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323,461 (1.23 MB)

 Trainable params: 322,565 (1.23 MB)

 Non-trainable params: 896 (3.50 KB)

In [47]:
checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(np.argmax(y_train,axis=1)),
    y=np.argmax(y_train,axis=1)
)

weights = dict(enumerate(weights))

history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=64,

    class_weight=weights,

    callbacks=[
        checkpoint,
        early,
        reduce
    ]

)

Epoch 1/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 345ms/step - accuracy: 0.6968 - loss: 0.7180
Epoch 1: val_accuracy improved from None to 0.72584, saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 405s 364ms/step - accuracy: 0.7692 - loss: 0.5581 - val_accuracy: 0.7258 - val_loss: 0.7488 - learning_rate: 0.0010
Epoch 2/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.8501 - loss: 0.3795
Epoch 2: val_accuracy improved from 0.72584 to 0.89492, saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 449s 370ms/step - accuracy: 0.8605 - loss: 0.3629 - val_accuracy: 0.8949 - val_loss: 0.3035 - learning_rate: 0.0010
Epoch 3/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 348ms/step - accuracy: 0.8832 - lo

In [48]:
model.load_weights(BEST_MODEL_PATH)

loss,accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy :",accuracy)

684/684 ━━━━━━━━━━━━━━━━━━━━ 30s 44ms/step - accuracy: 0.9669 - loss: 0.0961
Accuracy : 0.96690434217453


In [54]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(np.argmax(y_train, axis=1)),
    y=np.argmax(y_train, axis=1)
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(27.387793427230047), 1: np.float64(0.24185069511622123), 2: np.float64(2.6101118568232664), 3: np.float64(8.090984743411928), 4: np.float64(3.1057320319432122)}


In [56]:
checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[checkpoint, early, reduce]
)

Epoch 1/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.9639 - loss: 0.0638
Epoch 1: val_accuracy improved from None to 0.96423, saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 390s 357ms/step - accuracy: 0.9638 - loss: 0.0615 - val_accuracy: 0.9642 - val_loss: 0.1030 - learning_rate: 4.0000e-05
Epoch 2/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.9649 - loss: 0.0559
Epoch 2: val_accuracy improved from 0.96423 to 0.96583, saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/SmartECG-HD/models/best_model.keras
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 390s 357ms/step - accuracy: 0.9645 - loss: 0.0578 - val_accuracy: 0.9658 - val_loss: 0.0997 - learning_rate: 4.0000e-05
Epoch 3/20
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.9

In [57]:
import joblib

joblib.dump(
    encoder,
    MODEL_DIR + "/label_encoder.pkl"
)

['/content/drive/MyDrive/SmartECG-HD/models/label_encoder.pkl']

In [58]:
import joblib

joblib.dump(
    history.history,
    HISTORY_PATH
)

['/content/drive/MyDrive/SmartECG-HD/models/history.pkl']

In [59]:
model.load_weights(BEST_MODEL_PATH)

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(f"Test Accuracy: {accuracy:.4f}")

684/684 ━━━━━━━━━━━━━━━━━━━━ 31s 45ms/step - accuracy: 0.9678 - loss: 0.0946
Test Accuracy: 0.9678


In [60]:
import os

print(os.listdir(MODEL_DIR))
#best_model.keras
#history.pkl
# label_encoder.pkl

['best_model.keras', 'label_encoder.pkl', 'history.pkl']
